In [1]:
%pip install -q torch tokenizers numpy

You should consider upgrading via the '/Users/zade/Test/microGPT/model/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from tokenizers import ByteLevelBPETokenizer
import os
import torch

In [3]:

tokenizer = ByteLevelBPETokenizer(
    "tokenizer/vocab.json",
    "tokenizer/merges.txt"
)

In [4]:
from pathlib import Path

DATA_PATH = Path("dataset/wikipedia_text.txt")
OUTPUT_DIR = Path("dataset")
BATCH_SIZE = 512
BLOCK_SIZE = 256

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH}")

# Count lines without loading the dataset into memory.
with DATA_PATH.open("r", encoding="utf-8") as file:
    num_lines = sum(1 for _ in file)

train_end = int(num_lines * 0.8)
val_end = int(num_lines * 0.9)

print(f"Lines: {num_lines:,}")
print(f"Split boundaries: train={train_end:,}, validation={val_end - train_end:,}, test={num_lines - val_end:,}")

Lines: 42,350,660
Split boundaries: train=33,880,528, validation=4,235,066, test=4,235,066


In [5]:
import json
import numpy as np


def write_batch(batch, output):
    if not batch:
        return 0

    encodings = tokenizer.encode_batch(batch)
    ids = np.asarray(
        [token_id for encoding in encodings for token_id in encoding.ids],
        dtype=np.uint16,
    )
    ids.tofile(output)
    batch.clear()
    return int(ids.size)


split_info = {
    "train": {"start": 0, "end": train_end, "lines": 0, "tokens": 0},
    "val": {"start": train_end, "end": val_end, "lines": 0, "tokens": 0},
    "test": {"start": val_end, "end": num_lines, "lines": 0, "tokens": 0},
}
batches = {split_name: [] for split_name in split_info}

# A single sequential pass avoids keeping the corpus or encodings in memory.
with DATA_PATH.open("r", encoding="utf-8") as source:
    outputs = {
        split_name: (OUTPUT_DIR / f"{split_name}.bin").open("wb")
        for split_name in split_info
    }
    try:
        for line_number, line in enumerate(source):
            split_name = next(
                name for name, info in split_info.items()
                if info["start"] <= line_number < info["end"]
            )
            info = split_info[split_name]
            batches[split_name].append(line.rstrip("\\n"))
            info["lines"] += 1

            if len(batches[split_name]) >= BATCH_SIZE:
                info["tokens"] += write_batch(
                    batches[split_name], outputs[split_name]
                )

        for split_name, batch in batches.items():
            split_info[split_name]["tokens"] += write_batch(
                batch, outputs[split_name]
            )
    finally:
        for output in outputs.values():
            output.close()

metadata = {
    "vocab_size": tokenizer.get_vocab_size(),
    "source": DATA_PATH.name,
    "dtype": "uint16",
    "block_size": BLOCK_SIZE,
    "batch_size": BATCH_SIZE,
    "splits": {},
}

for split_name, info in split_info.items():
    metadata["splits"][split_name] = {
        "path": f"{split_name}.bin",
        "lines": info["lines"],
        "tokens": info["tokens"],
    }
    print(f"Saved {split_name}: {info['tokens']:,} tokens")

with (OUTPUT_DIR / "metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print(f"Saved metadata to {OUTPUT_DIR / 'metadata.json'}")

Saved train: 676,515,884 tokens
Saved val: 77,035,033 tokens
Saved test: 77,095,306 tokens
Saved metadata to dataset/metadata.json


In [6]:
def load_token_ids(split_name):
    """Memory-map a split so training never loads the whole file into RAM."""
    split = metadata["splits"][split_name]
    return np.memmap(
        OUTPUT_DIR / split["path"],
        dtype=np.uint16,
        mode="r",
        shape=(split["tokens"],),
    )


def get_batch(split_name, batch_size, device="cpu"):
    """Sample next-token-training batches from a memory-mapped split."""
    token_ids = load_token_ids(split_name)
    if len(token_ids) <= BLOCK_SIZE:
        raise ValueError(f"{split_name} needs more than {BLOCK_SIZE} tokens")

    starts = np.random.randint(0, len(token_ids) - BLOCK_SIZE - 1, size=batch_size)
    inputs = np.stack([token_ids[start:start + BLOCK_SIZE] for start in starts])
    targets = np.stack([token_ids[start + 1:start + BLOCK_SIZE + 1] for start in starts])

    return (
        torch.from_numpy(inputs.astype(np.int64)).to(device),
        torch.from_numpy(targets.astype(np.int64)).to(device),
    )


train_batch, target_batch = get_batch("train", batch_size=4)
print(f"Example input batch shape: {tuple(train_batch.shape)}")
print(f"Example target batch shape: {tuple(target_batch.shape)}")
print(f"Training files are ready in: {OUTPUT_DIR}/")

Example input batch shape: (4, 256)
Example target batch shape: (4, 256)
Training files are ready in: dataset/
